# Generative AI & LLMs: Interview Prep

This notebook explores key concepts in generative AI and large language models:
- Temperature and sampling strategies (top-k, top-p/nucleus sampling)
- Token probability and perplexity
- Autoregressive text generation
- Scaling laws
- Chain-of-thought prompting
- Cost estimation
- LLM architecture comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import softmax
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries loaded successfully!")

## 1. Temperature and Softmax

Temperature controls the "sharpness" of the probability distribution. Higher temperature = more random, lower = more deterministic.

In [ ]:
def temperature_sampling(logits, temperature=1.0):
    """
    Apply temperature to logits and compute probabilities.
    
    Args:
        logits: raw model outputs (unnormalized scores)
        temperature: controls distribution sharpness
    
    Returns:
        probabilities after softmax with temperature
    """
    scaled_logits = logits / temperature
    return softmax(scaled_logits)

# Simulate logits from a language model predicting next token
logits = np.array([2.0, 1.5, 1.0, 0.5, 0.1])
token_names = ['the', 'a', 'an', 'this', 'that']

# Compare different temperatures
temperatures = [0.3, 1.0, 2.0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, temp in zip(axes, temperatures):
    probs = temperature_sampling(logits, temperature=temp)
    ax.bar(token_names, probs, color='steelblue', alpha=0.7)
    ax.set_ylabel('Probability')
    ax.set_title(f'Temperature = {temp}')
    ax.set_ylim([0, 1])
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f'{p:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('temperature_sampling.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nTemperature Effect on {token_names}:")
for temp in temperatures:
    probs = temperature_sampling(logits, temperature=temp)
    print(f"\nTemperature {temp}:")
    for name, prob in zip(token_names, probs):
        print(f"  {name:6s}: {prob:.4f}")

## 2. Top-K Sampling

Top-K sampling restricts sampling to the K most likely tokens, eliminating low-probability tail.

In [ ]:
def top_k_sampling(logits, k=2, temperature=1.0):
    """
    Sample from top-k most likely tokens.
    
    Args:
        logits: raw model outputs
        k: number of top tokens to consider
        temperature: controls sharpness
    
    Returns:
        probabilities with top-k filtering applied
    """
    # Apply temperature
    probs = temperature_sampling(logits, temperature)
    
    # Find top-k indices
    top_k_indices = np.argsort(probs)[-k:]
    
    # Zero out non-top-k probabilities
    filtered_probs = np.zeros_like(probs)
    filtered_probs[top_k_indices] = probs[top_k_indices]
    
    # Renormalize
    return filtered_probs / filtered_probs.sum()

# Demonstrate top-k with different k values
k_values = [2, 3, 5]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, k in zip(axes, k_values):
    probs = top_k_sampling(logits, k=k, temperature=1.0)
    colors = ['green' if p > 0 else 'lightgray' for p in probs]
    ax.bar(token_names, probs, color=colors, alpha=0.7)
    ax.set_ylabel('Probability')
    ax.set_title(f'Top-K Sampling (K={k})')
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('top_k_sampling.png', dpi=100, bbox_inches='tight')
plt.show()

print("Top-K Sampling Results:")
for k in k_values:
    probs = top_k_sampling(logits, k=k)
    active_tokens = [(name, prob) for name, prob in zip(token_names, probs) if prob > 0]
    print(f"\nK={k} (Active tokens: {len(active_tokens)}):")
    for name, prob in active_tokens:
        print(f"  {name:6s}: {prob:.4f}")

## 3. Top-P (Nucleus) Sampling

Nucleus sampling keeps the minimum set of tokens that reach probability threshold p.

In [ ]:
def nucleus_sampling(logits, p=0.9, temperature=1.0):
    """
    Sample from nucleus (top-p) of probability distribution.
    
    Args:
        logits: raw model outputs
        p: cumulative probability threshold
        temperature: controls sharpness
    
    Returns:
        probabilities with nucleus filtering applied
    """
    # Apply temperature
    probs = temperature_sampling(logits, temperature)
    
    # Sort probabilities in descending order
    sorted_indices = np.argsort(-probs)
    sorted_probs = probs[sorted_indices]
    
    # Find minimum set of tokens with cumulative prob >= p
    cumsum = np.cumsum(sorted_probs)
    nucleus_mask = cumsum <= p
    nucleus_mask[np.argmax(cumsum >= p)] = True  # Always include the token pushing over p
    
    # Zero out non-nucleus tokens
    sorted_probs[~nucleus_mask] = 0
    
    # Reconstruct original order and renormalize
    filtered_probs = np.zeros_like(probs)
    filtered_probs[sorted_indices] = sorted_probs
    
    return filtered_probs / filtered_probs.sum()

# Demonstrate nucleus sampling
p_values = [0.5, 0.75, 0.95]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, p in zip(axes, p_values):
    probs = nucleus_sampling(logits, p=p, temperature=1.0)
    colors = ['orange' if prob > 0 else 'lightgray' for prob in probs]
    ax.bar(token_names, probs, color=colors, alpha=0.7)
    ax.set_ylabel('Probability')
    ax.set_title(f'Nucleus Sampling (P={p})')
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('nucleus_sampling.png', dpi=100, bbox_inches='tight')
plt.show()

print("Nucleus (Top-P) Sampling Results:")
for p in p_values:
    probs = nucleus_sampling(logits, p=p)
    active_tokens = [(name, prob) for name, prob in zip(token_names, probs) if prob > 0]
    cumsum_prob = sum(prob for _, prob in active_tokens)
    print(f"\nP={p} (Cumulative prob: {cumsum_prob:.4f}):")
    for name, prob in active_tokens:
        print(f"  {name:6s}: {prob:.4f}")

## 4. Perplexity

Perplexity measures how well a language model predicts a sequence. Lower perplexity = better model.

In [ ]:
def calculate_perplexity(log_probabilities):
    """
    Calculate perplexity from log probabilities.
    Perplexity = exp(-mean(log_prob))
    
    Args:
        log_probabilities: array of log probabilities for each token
    
    Returns:
        perplexity value
    """
    mean_log_prob = np.mean(log_probabilities)
    return np.exp(-mean_log_prob)

# Simulate token probabilities for different models
# Good model: high token probabilities
good_model_probs = np.array([0.8, 0.75, 0.85, 0.7, 0.9])
good_model_log_probs = np.log(good_model_probs)

# Medium model: medium token probabilities
medium_model_probs = np.array([0.5, 0.45, 0.55, 0.4, 0.6])
medium_model_log_probs = np.log(medium_model_probs)

# Poor model: low token probabilities
poor_model_probs = np.array([0.2, 0.15, 0.25, 0.1, 0.3])
poor_model_log_probs = np.log(poor_model_probs)

good_perplexity = calculate_perplexity(good_model_log_probs)
medium_perplexity = calculate_perplexity(medium_model_log_probs)
poor_perplexity = calculate_perplexity(poor_model_log_probs)

# Visualize perplexity
models = ['Good Model', 'Medium Model', 'Poor Model']
perplexities = [good_perplexity, medium_perplexity, poor_perplexity]
colors_bar = ['green', 'orange', 'red']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(models, perplexities, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Perplexity (Lower is Better)', fontsize=12)
ax.set_title('Language Model Perplexity Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, perp in zip(bars, perplexities):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{perp:.2f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('perplexity.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nPerplexity Results:")
print(f"Good Model:   {good_perplexity:.4f}")
print(f"Medium Model: {medium_perplexity:.4f}")
print(f"Poor Model:   {poor_perplexity:.4f}")
print("\nInterpretation: Lower perplexity indicates the model assigns higher probability to correct tokens.")

## 5. Autoregressive Generation

Generate text token-by-token, where each token is sampled from the model's probability distribution given all previous tokens.

In [ ]:
def mock_language_model(input_tokens):
    """
    Mock LLM that returns logits based on input length.
    In reality, this would be a neural network.
    """
    vocab_size = 10
    # Simulate logits that depend on input length
    base_logits = np.random.randn(vocab_size)
    # Bias toward common tokens as sequence grows
    bias = len(input_tokens) * 0.1
    return base_logits + bias

def autoregressive_generation(max_length=10, temperature=1.0, top_k=None, top_p=None):
    """
    Generate text using autoregressive sampling.
    """
    vocab = ['<start>', 'the', 'cat', 'sat', 'on', 'a', 'mat', 'jumped', 'over', '<eos>']
    generated = [0]  # Start token
    
    for _ in range(max_length - 1):
        # Get model logits
        logits = mock_language_model(generated)
        
        # Apply sampling strategy
        if top_k:
            probs = top_k_sampling(logits, k=top_k, temperature=temperature)
        elif top_p:
            probs = nucleus_sampling(logits, p=top_p, temperature=temperature)
        else:
            probs = temperature_sampling(logits, temperature)
        
        # Sample next token
        next_token = np.random.choice(len(vocab), p=probs)
        generated.append(next_token)
        
        # Stop if end-of-sequence token
        if next_token == len(vocab) - 1:
            break
    
    return ' '.join([vocab[t] for t in generated])

print("Autoregressive Generation Examples:\n")
print("Temperature = 0.5 (Deterministic):")
for i in range(3):
    print(f"  {i+1}. {autoregressive_generation(temperature=0.5)}")

print("\nTemperature = 1.0 (Balanced):")
for i in range(3):
    print(f"  {i+1}. {autoregressive_generation(temperature=1.0)}")

print("\nTop-K Sampling (K=3):")
for i in range(3):
    print(f"  {i+1}. {autoregressive_generation(top_k=3)}")

print("\nNucleus Sampling (P=0.85):")
for i in range(3):
    print(f"  {i+1}. {autoregressive_generation(top_p=0.85)}")

## 6. Scaling Laws

Language model performance follows power-law scaling with model size and compute.

In [ ]:
# Simulate Chinchilla scaling laws
# Loss = a * (N^-alpha) + b
# where N is parameter count

# Realistic scaling parameters (approximate from research)
alpha = 0.07  # Scaling exponent
a = 1.69  # Normalization constant
b = 1.0  # Minimum loss

# Parameter counts (millions)
param_counts = np.logspace(1, 4, 50)  # 10M to 10B parameters

# Calculate loss (inverse of performance)
loss = a * (param_counts ** (-alpha)) + b

# Create scaling law visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Log-log plot
ax1.loglog(param_counts, loss, 'b-', linewidth=2.5, label='Power-law scaling')
ax1.scatter([100, 300, 1000, 3000], 
           [a * (x ** (-alpha)) + b for x in [100, 300, 1000, 3000]],
           s=100, c='red', alpha=0.7, zorder=5, label='Example models')
ax1.set_xlabel('Parameters (Millions)', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_title('Scaling Law: Loss vs Model Size (Log-Log)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Linear plot with annotations
ax2.plot(param_counts, loss, 'g-', linewidth=2.5)
ax2.scatter([100, 300, 1000, 3000], 
           [a * (x ** (-alpha)) + b for x in [100, 300, 1000, 3000]],
           s=100, c='red', alpha=0.7, zorder=5)
ax2.annotate('Small Model', xy=(100, a * (100 ** (-alpha)) + b), xytext=(100, 3),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)
ax2.annotate('Large Model', xy=(3000, a * (3000 ** (-alpha)) + b), xytext=(3000, 1.5),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)
ax2.set_xlabel('Parameters (Millions)', fontsize=11)
ax2.set_ylabel('Loss', fontsize=11)
ax2.set_title('Scaling Law: Loss vs Model Size (Linear)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('scaling_laws.png', dpi=100, bbox_inches='tight')
plt.show()

print("Scaling Law Analysis (Chinchilla)\n")
print("Parameters\tLoss")
print("-" * 30)
for params in [100, 300, 1000, 3000, 10000]:
    loss_val = a * (params ** (-alpha)) + b
    print(f"{params:,}M\t\t{loss_val:.4f}")
print(f"\nKey Insight: Doubling parameters improves loss by ~{((1/2)**alpha - 1)*100:.1f}%")

## 7. Chain-of-Thought Prompting

CoT prompting encourages models to reason step-by-step, improving accuracy on complex tasks.

In [ ]:
# Chain-of-Thought Prompt Examples

standard_prompt = """
Q: If a train travels at 60 mph for 2 hours, then 80 mph for 1 hour, 
what is the average speed?
A:
"""

cot_prompt = """
Q: If a train travels at 60 mph for 2 hours, then 80 mph for 1 hour, 
what is the average speed?

Let's think step by step:
1. Calculate distance in first leg: speed × time = 60 × 2 = 120 miles
2. Calculate distance in second leg: speed × time = 80 × 1 = 80 miles
3. Total distance: 120 + 80 = 200 miles
4. Total time: 2 + 1 = 3 hours
5. Average speed: total distance / total time = 200 / 3 ≈ 66.67 mph

A: The average speed is approximately 66.67 mph.
"""

print("Chain-of-Thought Prompting Comparison\n")
print("=" * 60)
print("STANDARD PROMPT (Direct Answer):")
print("-" * 60)
print(standard_prompt)
print("\nExpected Issue: Model may directly guess without reasoning.\n")

print("\n" + "=" * 60)
print("CHAIN-OF-THOUGHT PROMPT (Step-by-Step):")
print("-" * 60)
print(cot_prompt)
print("\nBenefit: Clear reasoning path improves accuracy and interpretability.\n")

# Simulate accuracy improvement
cot_scenarios = pd.DataFrame({
    'Task': ['Arithmetic', 'Logical Reasoning', 'Commonsense QA', 'Math Word Problems'],
    'Standard Accuracy': [0.65, 0.62, 0.71, 0.52],
    'CoT Accuracy': [0.92, 0.85, 0.81, 0.78]
})

cot_scenarios['Improvement'] = ((cot_scenarios['CoT Accuracy'] - cot_scenarios['Standard Accuracy']) / 
                                 cot_scenarios['Standard Accuracy'] * 100)

print("\nEmpirical CoT Impact on Accuracy:\n")
print(cot_scenarios.to_string(index=False))

# Visualize improvement
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(cot_scenarios))
width = 0.35

ax.bar(x - width/2, cot_scenarios['Standard Accuracy'], width, label='Standard', color='lightcoral', alpha=0.8)
ax.bar(x + width/2, cot_scenarios['CoT Accuracy'], width, label='Chain-of-Thought', color='lightgreen', alpha=0.8)

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Chain-of-Thought Prompting Impact on Task Accuracy', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(cot_scenarios['Task'], rotation=15, ha='right')
ax.legend(fontsize=11)
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# Add improvement labels
for i, improvement in enumerate(cot_scenarios['Improvement']):
    ax.text(i, cot_scenarios['CoT Accuracy'].iloc[i] + 0.03, 
           f'+{improvement:.1f}%', ha='center', fontsize=10, fontweight='bold', color='green')

plt.tight_layout()
plt.savefig('chain_of_thought.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Cost Estimation for LLM Inference

Estimate inference costs based on token usage and pricing models.

In [ ]:
# LLM pricing models (as of early 2024, approximate)
pricing_data = {
    'Model': ['GPT-3.5', 'GPT-4', 'Claude 2', 'Llama 2 (Open)'],
    'Input Cost ($/1M tokens)': [0.50, 30.00, 8.00, 0.00],
    'Output Cost ($/1M tokens)': [1.50, 60.00, 24.00, 0.00],
    'Context Window': [4096, 8192, 100000, 4096],
}

pricing_df = pd.DataFrame(pricing_data)

print("LLM Pricing Comparison\n")
print(pricing_df.to_string(index=False))

def estimate_cost(model_name, input_tokens, output_tokens, pricing_df):
    """
    Estimate inference cost for a given model.
    """
    row = pricing_df[pricing_df['Model'] == model_name].iloc[0]
    input_cost = (input_tokens / 1_000_000) * row['Input Cost ($/1M tokens)']
    output_cost = (output_tokens / 1_000_000) * row['Output Cost ($/1M tokens)']
    return input_cost + output_cost

# Scenarios
scenarios = [
    ('Short Query', 100, 50),
    ('Medium Query', 500, 200),
    ('Long Document', 5000, 1000),
    ('Large Batch (1000x medium)', 500_000, 200_000)
]

print("\n\nCost Estimation for Different Scenarios:\n")

for scenario_name, in_tokens, out_tokens in scenarios:
    print(f"\n{scenario_name} (Input: {in_tokens:,} | Output: {out_tokens:,} tokens):")
    print("-" * 50)
    for model in pricing_df['Model']:
        cost = estimate_cost(model, in_tokens, out_tokens, pricing_df)
        print(f"  {model:20s}: ${cost:.6f}")

# Visualize cost comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (scenario_name, in_tokens, out_tokens) in enumerate(scenarios):
    costs = [estimate_cost(model, in_tokens, out_tokens, pricing_df) 
            for model in pricing_df['Model']]
    
    ax = axes[idx]
    bars = ax.bar(pricing_df['Model'], costs, color=['#FF6B6B', '#FF4C4C', '#FFB347', '#90EE90'], alpha=0.8)
    ax.set_ylabel('Cost ($)', fontsize=11)
    ax.set_title(scenario_name, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, cost in zip(bars, costs):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'${cost:.4f}' if cost > 0 else 'Free',
               ha='center', va='bottom', fontsize=9)

plt.suptitle('LLM Inference Cost Comparison Across Scenarios', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('cost_estimation.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. LLM Architecture Comparison

Key differences between major language model architectures.

In [ ]:
# Comprehensive LLM architecture comparison
arch_comparison = pd.DataFrame({
    'Architecture': ['GPT (Decoder-Only)', 'Claude (Decoder-Only)', 'LLaMA (Decoder-Only)', 
                     'T5 (Encoder-Decoder)', 'BERT (Encoder-Only)'],
    'Type': ['Decoder-Only', 'Decoder-Only', 'Decoder-Only', 'Encoder-Decoder', 'Encoder-Only'],
    'Primary Task': ['Text Generation', 'Text Generation', 'Text Generation', 
                     'Seq2Seq Tasks', 'Classification/Tagging'],
    'Context Window': ['4K-128K', '100K', '4K', '512', '512'],
    'Training Objective': ['Next Token Prediction', 'Next Token Prediction', 'Next Token Prediction',
                          'Denoising', 'MLM + NSP'],
    'Causal Attention': ['Yes', 'Yes', 'Yes', 'No (Encoder)', 'No'],
    'Parameter Sharing': ['Full', 'Full', 'Full', 'Some', 'Full']
})

print("\nLLM Architecture Comparison\n")
print("=" * 120)
print(arch_comparison.to_string(index=False))

print("\n\nKey Architecture Details:\n")

arch_details = {
    'Decoder-Only (GPT, Claude, LLaMA)': [
        '• Best for: Open-ended text generation and few-shot learning',
        '• Causal attention: Cannot see future tokens (autoregressive)',
        '• Training: Next token prediction objective',
        '• Versatility: Can handle many tasks via prompting',
        '• Example: LLM chat, code generation, question answering'
    ],
    'Encoder-Decoder (T5, BART)': [
        '• Best for: Translation, summarization, structured tasks',
        '• Encoder: Bidirectional context, can see full sequence',
        '• Decoder: Generates output autoregressively',
        '• Flexibility: Separate encoding/decoding allows specialization',
        '• Example: Machine translation, text summarization'
    ],
    'Encoder-Only (BERT, RoBERTa)': [
        '• Best for: Classification, token-level tasks, embeddings',
        '• Bidirectional attention: Uses context from both sides',
        '• Training: Masked language modeling (MLM)',
        '• Limitation: Cannot generate text directly',
        '• Example: Sentiment analysis, entity recognition, similarity'
    ]
}

for arch_type, details in arch_details.items():
    print(f"{arch_type}:")
    for detail in details:
        print(f"  {detail}")
    print()

## Summary

Key takeaways:
1. **Temperature**: Controls randomness in sampling; lower = more deterministic, higher = more creative
2. **Top-K & Nucleus Sampling**: Limit sampling to most likely tokens, avoiding nonsensical outputs
3. **Perplexity**: Measures model quality on a test set; lower is better
4. **Autoregressive Generation**: Models predict one token at a time given previous context
5. **Scaling Laws**: Model performance improves predictably with size (power-law relationship)
6. **Chain-of-Thought**: Step-by-step reasoning significantly improves task accuracy
7. **Cost Analysis**: Important for deployment decisions; API costs vary dramatically
8. **Architecture Choice**: Decoder-only best for generation; encoder-decoder for structured tasks; encoder-only for classification

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>